# Project 2

ST554: Analysis of Big Data

Ryan Mersereau

## Part 1: Testing our custom Class `SparkDataCheck` on Air Quality Data

### Introduction

In this notebook, we'll start by using our custom `SparkDataCheck` Python class. We'll use it on the **UCI Air Quality dataset** from project 1, which records hourly averaged
sensor measurements of several pollutants (CO, NOx, NO₂, Benzene, O₃, etc.) in an Italian city over roughly one year (March 2004 - April 2005).


### Setup

First, we will reload the python class, import other needed modules, and start a spark session.


In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Project2") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 22:04:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 22:04:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 22:04:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/26 22:04:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/26 22:04:48 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/03/26 22:04:48 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [2]:
import importlib
import my_class
importlib.reload(my_class)

<module 'my_class' from '/home/jupyter-ramerser@ncsu.edu/my_class.py'>

### Load Data 

Let's read the air quality CSV directly from the csv filepath into a `SparkDataCheck`
object using the `from_csv()` class method.  

The dataset contains **9,357 rows** and **15 columns**. Missing values are recorded as
`-200` rather than `NULL`, which we will look at later on.


In [3]:
CSV_PATH = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

air_pandas = pd.read_csv(CSV_PATH)
air_pandas.to_csv("air.csv", index=False)

# Load from local path, saved as air_check (to check our class on)
air_check = my_class.SparkDataCheck.from_csv(spark, "air.csv")

# Drop first column (index)
air_check.df = air_check.df.drop("Unnamed: 0")

# Look at the schema and first few rows
air_check.df.printSchema()
air_check.df.show(5, truncate=False)


root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Date     |Time               |CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|T   |RH  |AH    |
+---------+-------------------+------+-----------+--------+--------+-------------+-----

###  Validation Methods

Now that the data is loaded in correctly, lets check some of our validation methods!

First, lets look at  

#### `check_numeric_range()`

This method appends a Boolean column indicating whether each
value falls within the supplied range (inclusive). NULL values will appear as NULL.

**Example 1 – Both bounds, CO sensor:** From the UCI Air quality website, acceptable CO(GT) readings should sit
between 0 and 11 mg/m³, so we can see if the first 10 observations fall within this range.


In [4]:
# Example 1: both lower and upper bounds on a numeric column
air_check.check_numeric_range("CO(GT)", lower=0, upper=11)
air_check.df.select("CO(GT)", "CO(GT)_in_bounds").show(10)


+------+----------------+
|CO(GT)|CO(GT)_in_bounds|
+------+----------------+
|   2.6|            true|
|   2.0|            true|
|   2.2|            true|
|   2.2|            true|
|   1.6|            true|
|   1.2|            true|
|   1.2|            true|
|   1.0|            true|
|   0.9|            true|
|   0.6|            true|
+------+----------------+
only showing top 10 rows


Here we can see the first 10 observations do fall within acceptable bounds, so this method seems to be working

**Example 2 – Lower bound only:**  We can also check whether temperature (`T`) is at least −5°C, and flag any suspiciously cold readings.


In [5]:
# Example 2: lower bound only
air_check.check_numeric_range("T", lower=-5)
air_check.df.select("T", "T_in_bounds").show(10)


+----+-----------+
|   T|T_in_bounds|
+----+-----------+
|13.6|       true|
|13.3|       true|
|11.9|       true|
|11.0|       true|
|11.2|       true|
|11.2|       true|
|11.3|       true|
|10.7|       true|
|10.7|       true|
|10.3|       true|
+----+-----------+
only showing top 10 rows


**Example 3 – Upper bound only:** Relative humidity (`RH`) shouldn't exceed 100%, so we can check to see if our datapoints are below that threshold.


In [6]:
# Example 3: upper bound only
air_check.check_numeric_range("RH", upper=100)
air_check.df.select("RH", "RH_in_bounds").show(10)


+----+------------+
|  RH|RH_in_bounds|
+----+------------+
|48.9|        true|
|47.7|        true|
|54.0|        true|
|60.0|        true|
|59.6|        true|
|59.2|        true|
|56.8|        true|
|60.0|        true|
|59.7|        true|
|60.2|        true|
+----+------------+
only showing top 10 rows


**Example 4 – No bounds supplied (triggers warning message):**
Calling the method without providing *either* bound should print an informative
message and leave the DataFrame unchanged.


In [7]:
# Example 4: neither bound provided
air_check.check_numeric_range("NOx(GT)")


check_numeric_range: please supply at least one of 'lower' or 'upper'.


**Example 5 – Non-numeric column supplied (triggers warning message):**
Passing the string-typed `Date` column should print a message and return the original
object.


In [8]:
# Example 5: non-numeric column
air_check.check_numeric_range("Date", lower=0, upper=100)
print("Columns after invalid call:", air_check.df.columns)


check_numeric_range: column 'Date' has type 'string' which is not numeric. Returning original DataFrame.
Columns after invalid call: ['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'CO(GT)_in_bounds', 'T_in_bounds', 'RH_in_bounds']


No new column was added because the 'Date' column is a string, not numeric

Now, let's look at 5 examples of our next method:

#### `check_levels()`

This method appends a Boolean column indicating whether each value in a *string*
column belongs to a defined set of levels. We can use the `Date` and `Time` columns to
demonstrate this method.

**Example 1 – Valid months in the Date column:**
The dataset runs from March 2004 to February 2005. We check whether the month
portion of the date falls within a known good set.


In [9]:
# Example 1: check Date values against a set of known dates in the dataset
# Add a Month string column so we have a clean category to check
air_check.df = air_check.df.withColumn("Month", F.split(F.col("Date"), "/")[0]) # Splitting by / character, first portion is the month

valid_months = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]
air_check.check_levels("Month", levels=valid_months)
air_check.df.select("Date", "Month", "Month_in_levels").show(10)


+---------+-----+---------------+
|     Date|Month|Month_in_levels|
+---------+-----+---------------+
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
+---------+-----+---------------+
only showing top 10 rows


Here a new 'Month' column is created, which is subsetted from Date

**Example 2 – Subset of valid months (some will be False):**
We can also subset to only flag dates from the summer months (June–August) as valid.


In [10]:
# Example 2: restrict to summer months – many rows will be False
air_check.check_levels("Month", levels=["6","7","8"])
air_check.df.select("Date", "Month", "Month_in_levels").show(10)


+---------+-----+---------------+
|     Date|Month|Month_in_levels|
+---------+-----+---------------+
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
+---------+-----+---------------+
only showing top 10 rows


For these first 10 observations, March (month 3) is not a in the valid specified summer month, so the levels column returns false

**Example 3 – Check Time values against expected hours:**
Since the data is hourly; valid hours run 00–23, we can check if hourly timestamps of our data are valid


In [11]:
# Example 3: check Time column against expected hour strings
air_check.df = air_check.df.withColumn("Time", F.date_format(F.col("Time"), "HH:mm:ss"))

valid_hours = [f"{h:02d}:00:00" for h in range(24)]
air_check.check_levels("Time", levels=valid_hours)
air_check.df.select("Time", "Time_in_levels").show(10)


+--------+--------------+
|    Time|Time_in_levels|
+--------+--------------+
|18:00:00|          true|
|19:00:00|          true|
|20:00:00|          true|
|21:00:00|          true|
|22:00:00|          true|
|23:00:00|          true|
|00:00:00|          true|
|01:00:00|          true|
|02:00:00|          true|
|03:00:00|          true|
+--------+--------------+
only showing top 10 rows


Above, we modify the 'Time' column to represent the hour of each observation from the original 'Time' column in the Dataframe 

**Example 4 – Non-string column supplied (triggers warning message):**
Passing the integer column `PT08.S1(CO)` should print a warning message and return the
object unmodified.


In [12]:
# Example 4: non-string column 
air_check.check_levels("PT08.S1(CO)", levels=["good","bad"]) # Given this column, was a string, we could use levels like good/bad or high/low


check_categorical_levels: column 'PT08.S1(CO)' has type 'int' which is not a string. DataFrame is unchanged.


**Example 5 – Method chaining across two categorical checks:**
Finally, we can chain the call on `Month` and `Time` in a single expression, because each
method returns `self`. Let's try to check if a month is in summer and if the time is during 12 AM to 2 AM, and return boolean indicators for each.


In [52]:
# Example 5: chained categorical checks
air_check \
    .check_levels("Month", levels=["6","7","8"]) \
    .check_levels("Time", levels=["00:00:00","01:00:00","02:00:00"])

air_check.df.select("Date","Time","Month_in_levels","Time_in_levels").show(10)


+---------+--------+---------------+--------------+
|     Date|    Time|Month_in_levels|Time_in_levels|
+---------+--------+---------------+--------------+
|3/10/2004|18:00:00|          false|         false|
|3/10/2004|19:00:00|          false|         false|
|3/10/2004|20:00:00|          false|         false|
|3/10/2004|21:00:00|          false|         false|
|3/10/2004|22:00:00|          false|         false|
|3/10/2004|23:00:00|          false|         false|
|3/11/2004|00:00:00|          false|          true|
|3/11/2004|01:00:00|          false|          true|
|3/11/2004|02:00:00|          false|          true|
|3/11/2004|03:00:00|          false|         false|
+---------+--------+---------------+--------------+
only showing top 10 rows


Now, lets look at 5 examples of

#### `check_missing()` 

This method appends a Boolean column that is `True` wherever the source column is
`NULL`. It works on *any* column type.


##### Brief Data Cleaning

Since null values are recorded as `-200` in this data, let's replace them with null to accurately use our `check_missing()` method


In [14]:
air_check.df = air_check.df.na.replace(-200.0, None)
air_check.df = air_check.df.na.replace(-200, None)

# To confirm it worked, this should show a null
air_check.df.select("CO(GT)", "NOx(GT)", "T").show(10)

+------+-------+----+
|CO(GT)|NOx(GT)|   T|
+------+-------+----+
|   2.6|    166|13.6|
|   2.0|    103|13.3|
|   2.2|    131|11.9|
|   2.2|    172|11.0|
|   1.6|    131|11.2|
|   1.2|     89|11.2|
|   1.2|     62|11.3|
|   1.0|     62|10.7|
|   0.9|     45|10.7|
|   0.6|   NULL|10.3|
+------+-------+----+
only showing top 10 rows


**Example 1 – Missing CO sensor readings:**

In [15]:
# Example 1: check for NULLs in CO(GT)
air_check.check_missing("CO(GT)")
air_check.df.select("CO(GT)", "CO(GT)_is_missing").show(10)


+------+-----------------+
|CO(GT)|CO(GT)_is_missing|
+------+-----------------+
|   2.6|            false|
|   2.0|            false|
|   2.2|            false|
|   2.2|            false|
|   1.6|            false|
|   1.2|            false|
|   1.2|            false|
|   1.0|            false|
|   0.9|            false|
|   0.6|            false|
+------+-----------------+
only showing top 10 rows


**Example 2 – Missing temperature readings:**


In [16]:
# Example 2: missing values in T (temperature)
air_check.check_missing("T")
air_check.df.select("T", "T_is_missing").show(10)


+----+------------+
|   T|T_is_missing|
+----+------------+
|13.6|       false|
|13.3|       false|
|11.9|       false|
|11.0|       false|
|11.2|       false|
|11.2|       false|
|11.3|       false|
|10.7|       false|
|10.7|       false|
|10.3|       false|
+----+------------+
only showing top 10 rows


**Example 3 – Missing values in a string column (`Date`):**
The method accepts any column type, so we can use it on `Date` as well.


In [17]:
# Example 3: missing Date values
air_check.check_missing("Date")
air_check.df.select("Date", "Date_is_missing").show(10)


+---------+---------------+
|     Date|Date_is_missing|
+---------+---------------+
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
+---------+---------------+
only showing top 10 rows


Since no null values are present in the top 10 rows for CO, Temp, and Date, let's see if we can use this method to find the total count of nulls.

**Example 4 – Count the proportion of missing values for a column:**
Once the Boolean column exists, standard Spark aggregation can tell us the
missing rate!


In [18]:
# Example 4: compute missing rate for AH (absolute humidity)
air_check.check_missing("AH")

total = air_check.df.count()
missing = air_check.df.filter(F.col("AH_is_missing")).count()
print(f"AH: {missing}/{total} rows are NULL ({100*missing/total:.1f}%)")


AH: 366/9357 rows are NULL (3.9%)


As we can see above, 366 observations of absolute humidity, or 3.9% of total rows are null values which could be due to errors or outages in the sensor technology.

**Example 5 – Chaining `check_missing` with `check_numeric_bounds`:**
Like we did earlier, we can also mix validation methods in a single chain. Lets try to use our methods to check for missing values, and then the range of those values.


In [19]:
# Example 5: chain missing-check with a bounds-check on NOx
air_check \
    .check_missing("NOx(GT)") \
    .check_numeric_range("NOx(GT)", lower=0, upper=1000)

air_check.df.select("NOx(GT)", "NOx(GT)_is_missing", "NOx(GT)_in_bounds").show(10)


+-------+------------------+-----------------+
|NOx(GT)|NOx(GT)_is_missing|NOx(GT)_in_bounds|
+-------+------------------+-----------------+
|    166|             false|             true|
|    103|             false|             true|
|    131|             false|             true|
|    172|             false|             true|
|    131|             false|             true|
|     89|             false|             true|
|     62|             false|             true|
|     62|             false|             true|
|     45|             false|             true|
|   NULL|              true|             NULL|
+-------+------------------+-----------------+
only showing top 10 rows


From above, the non-null values for NOx(GT) are all within the specified bounds of 0 to 1000. However, the null value also returns null when the check range method is called on it, since
null can't be evaluated as being inside of a range.

###  Summarization Methods

Now, lets test the summarization methods from our class. First, with 5 examples of:

####   `summarize_numeric()` 

This method returns a **pandas** DataFrame with the min and max of numeric
column(s), and has an optional `group_by` argument.

**Example 1 – Single column, no grouping:**


In [20]:
# Example 1: min/max of CO(GT) – no grouping
air_check.summarize_numeric("CO(GT)")



,CO(GT)_min,CO(GT)_max
0,0.1,11.9


As we can see, the minimum CO concentration observed was 0.1mg, and the maximum was 11.9mg. Next, lets try and use the group_by argument for temperature grouped by month.

**Example 2 – Single column grouped by Month:**


In [21]:
# Example 2: min/max of temperature grouped by month
air_check.summarize_numeric("T", group_by="Month")



,Month,T_min,T_max
0,1,1.0,16.9
1,10,12.6,30.8
2,11,1.3,26.8
3,12,1.2,20.3
4,2,0.3,19.9
5,3,-1.9,29.3
6,4,7.4,31.3
7,5,7.4,32.8
8,6,16.5,42.2
9,7,16.2,44.6


The month output is not in chronological order (something to fix), but we can see how minimum and maximum temperature varies greatly throughout the year.

**Example 3 – All numeric columns, no grouping:**
When no column is supplied, the method finds and summarizes every numeric column
in a single wide pandas DataFrame.


In [22]:
# Example 3: all numeric columns at once
air_check.summarize_numeric()



26/03/26 22:04:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,PT08.S2(NMHC)_min,PT08.S2(NMHC)_max,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,0.1,11.9,647,2040,7,1189,0.1,63.7,383,2214,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


**Example 4 – All numeric columns grouped by Month (`reduce` + `pd.merge` path):** Furthermore, we can group every column by a category such as month, and find the minimum and maximum values corresponding to each variable by month.


In [23]:
# Example 4: all numeric columns grouped by month
air_check.summarize_numeric(group_by="Month")



,Month,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,PT08.S2(NMHC)_min,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,1,0.1,8.7,744,1835,NaN,NaN,0.1,43.0,383,...,657,2083,253,2331,1.0,16.9,17.2,86.6,0.2477,1.0567
1,10,0.4,9.5,704,1908,NaN,NaN,0.5,52.1,444,...,1050,2775,392,2372,12.6,30.8,31.8,86.5,0.9107,2.0224
2,11,0.1,11.9,647,2008,NaN,NaN,0.2,63.7,397,...,697,2643,261,2522,1.3,26.8,14.0,87.1,0.1988,1.9800
3,12,0.2,9.9,691,1881,NaN,NaN,0.4,50.8,426,...,682,2405,318,2523,1.2,20.3,16.3,88.7,0.2749,1.3713
4,2,0.3,8.4,755,1846,NaN,NaN,0.2,33.9,402,...,621,1870,252,2494,0.3,19.9,18.0,86.6,0.2269,1.0859
5,3,0.1,8.1,715,2040,7.0,797.0,0.2,39.2,387,...,551,2679,221,2359,-1.9,29.3,13.5,84.0,0.1847,1.3930
6,4,0.3,7.3,753,1875,9.0,1189.0,0.5,40.3,448,...,742,2684,263,2108,7.4,31.3,9.9,82.4,0.3866,1.4852
7,5,0.1,6.5,736,1763,275.0,275.0,0.5,40.2,437,...,1048,2667,301,2054,7.4,32.8,9.2,85.2,0.3754,1.6296
8,6,0.1,6.4,708,1506,NaN,NaN,0.5,36.9,437,...,1227,2746,334,2202,16.5,42.2,11.9,82.3,0.7502,1.9390
9,7,0.1,5.3,723,1626,NaN,NaN,1.3,37.3,527,...,1234,2662,386,2475,16.2,44.6,9.6,69.3,0.7158,2.0042


**Example 5 – Non-numeric column supplied (triggers warning, returns None):** This is useful because we don't want to perform summary statistics on a number that is nominal, but we also don't want
it's input to cause errors.


In [53]:
# Example 5: non-numeric column – should print a warning and return None
air_check.summarize_numeric("Date")

summarize_numeric: column 'Date' has type 'string' which is not numeric. Returning None.


Next, let's look at 5 examples of:

####  `summarize_counts()` 

This method returns a **pandas** DataFrame of value counts for one or two string columns.

**Example 1 – Counts by Month:**


In [25]:
# Example 1: row counts per month
air_check.summarize_counts("Month")



,Month,count
0,1,744
1,10,744
2,11,720
3,12,744
4,2,672
5,3,1254
6,4,807
7,5,744
8,6,720
9,7,744


We see some disparity between the number of months, but since the experiment ran from mid-March 2004 to April 2005, data from march should have more observations than other months.

**Example 2 – Counts by Time (hour of day):**


In [26]:
# Example 2: row counts per hour
air_check.summarize_counts("Time")


,Time,count
0,00:00:00,390
1,01:00:00,390
2,02:00:00,390
3,03:00:00,390
4,04:00:00,390
5,05:00:00,390
6,06:00:00,390
7,07:00:00,390
8,08:00:00,390
9,09:00:00,390


Above time counts are very consistent, which makes sense as observations were systematically recorded hourly.

**Example 3 – Cross-tabulation: Month × Time:**
Now, we can examine using two string columns, every combination of levels for month and time is counted, essentially a
cross-tabulation.


In [27]:
# Example 3: counts for Month × Time combinations
result = air_check.summarize_counts("Month", column2="Time")
print(result.shape)
result.head(15)


(288, 3)


,Month,Time,count
0,1,00:00:00,31
1,1,01:00:00,31
2,1,02:00:00,31
3,1,03:00:00,31
4,1,04:00:00,31
5,1,05:00:00,31
6,1,06:00:00,31
7,1,07:00:00,31
8,1,08:00:00,31
9,1,09:00:00,31


This result makes sense as these counts are for each hour in a given month, and there are 31 days in the month of January. Its also worth noting that the shape of the result (288) = 12 months * 24 hours, validating accurate reading of data.

**Example 4 – Non-string primary column (returns None):**


In [28]:
# Example 4: numeric column as primary – should return None
air_check.summarize_counts("CO(GT)")



summarize_counts: column 'CO(GT)' has type 'double' which is not a string column.


**Example 5 – Non-string secondary column (returns None):**


In [29]:
# Example 5: valid string primary, but numeric secondary, also returns none
air_check.summarize_counts("Month", column2="T")


summarize_counts: column 'T' has type 'double' which is not a string column.


###  Load from Pandas – `from_pandas()`

Now, we can read the same dataset using plain pandas, then hand the pandas DataFrame
to `SparkDataCheck.from_pandas()`. This path calls `spark.createDataFrame()` to
convert the pandas DataFrame into a Spark DataFrame before wrapping it in our class.


In [30]:
# Read with pandas first
air_df = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
print("Pandas shape:", air_df.shape)
air_df.head()


Pandas shape: (9357, 16)


,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [31]:
# Create a SparkDataCheck instance from the pandas DataFrame
air_check_pd = my_class.SparkDataCheck.from_pandas(spark, air_df)

# Confirm the Spark DataFrame was created correctly
air_check_pd.df.printSchema()


root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): long (nullable = true)
 |-- NMHC(GT): long (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): long (nullable = true)
 |-- NOx(GT): long (nullable = true)
 |-- PT08.S3(NOx): long (nullable = true)
 |-- NO2(GT): long (nullable = true)
 |-- PT08.S4(NO2): long (nullable = true)
 |-- PT08.S5(O3): long (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



###  Example Method Call on the Pandas-Sourced Object

We run `summarize_numeric()` on `air_check_pd` to check the min/max of **all
numeric columns** and that the pandas-to-Spark conversion preserved the data correctly. We expect results that match what we saw from above.

It's worth noting here we didn't do any basic data cleaning (getting rid of first index column and replacing `-200` observations with null, but this could easily be done as this step is more for proof-of-concept.


In [32]:
# Single example: min/max of all numeric columns from the pandas-sourced object
summary_pd = air_check_pd.summarize_numeric()
summary_pd


,Unnamed: 0_min,Unnamed: 0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,0,9356,-200.0,11.9,-200,2040,-200,1189,-200.0,63.7,...,-200,2775,-200,2523,-200.0,44.6,-200.0,88.7,-200.0,2.231


`from_pandas()` is useful in practice because our plain pandas dataframe that is already processed can be handed to Spark, where we can then use our class. Essentially it "bridges the gap" between the two.

### Conclusion

Creating a python class with several methods is time-consuming, but incredibly adaptable and useful for being applied to a given Spark Dataframe. In this notebook we saw how our created class could be applied to the UCI air quality dataset from project 1 and used to validate and produce summary statistics of the data.

## Part 2: NFL Data Analysis

Now, we're going to switch gears and do some basic analysis using spark on some NFL data. 

### I. Pandas-on-Spark

#### Load Data

Let's start by using `pandas`-on-Spark to read in the weekly NFL data, and looking at the first 5 rows and all column names

In [33]:
import pyspark.pandas as ps

#Read in weekly NFL data
nfl_ps = ps.read_csv("weekly_nfl_data.csv")
nfl_ps.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


In [34]:
print(nfl_ps.columns.tolist())

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


This dataset has over **120,000** observations of player data by week over several decades, and recording nearly every noteworthy stat from that player. To do some analysis, we'll have to subset this massive dataset to focus on specific players, positions, and statistics.

#### Subsetting and Variable Creation

Now, lets look only at Quarterback stats for the seasons **2005-2023**. Additionally, subsetting the columns to only include player_display_name, season, week, completions,
attempts, passing_yards, passing_tds, and interceptions.

Then, for each player name and season combination, we want the find the sum and mean of each statistical quantity.

Finally, we'll create two new variables: Completion percentage and TD/Int ratio

In [35]:
# Subset rows: position == "QB", season_type == "REG", season between 2005 and 2023
nfl_qb_ps = nfl_ps[
    (nfl_ps["position"] == "QB") &
    (nfl_ps["season_type"] == "REG") &
    (nfl_ps["season"] >= 2005) &
    (nfl_ps["season"] <= 2023)
]

# Subset columns
nfl_qb_ps = nfl_qb_ps[[
    "player_display_name", "season", "week",
    "completions", "attempts", "passing_yards",
    "passing_tds", "interceptions"
]]

# For each player/season combination, find the sum and mean of each stat
nfl_agg_ps = nfl_qb_ps.groupby(["player_display_name", "season"]).agg(
    completions_sum=("completions", "sum"),
    completions_mean=("completions", "mean"),
    attempts_sum=("attempts", "sum"),
    attempts_mean=("attempts", "mean"),
    passing_yards_sum=("passing_yards", "sum"),
    passing_yards_mean=("passing_yards", "mean"),
    passing_tds_sum=("passing_tds", "sum"),
    passing_tds_mean=("passing_tds", "mean"),
    interceptions_sum=("interceptions", "sum"),
    interceptions_mean=("interceptions", "mean")
).reset_index()

# Create two new variables
nfl_agg_ps["completion_percentage"] = (
    nfl_agg_ps["completions_sum"] / nfl_agg_ps["attempts_sum"]
)
nfl_agg_ps["td_int_ratio"] = (
    nfl_agg_ps["passing_tds_sum"] / nfl_agg_ps["interceptions_sum"]
)

nfl_agg_ps.head()

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,0.666667,0.500000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN


Great, now we have a table with Quarterbacks fitting the year criteria. We see their sum and mean of each statistical quantity is displayed, along with out newly created variables of completion percentage and touchdown to interception ratio for that season. It's worth noting that 2007 Kerry Collins has a 'NaN' td/int ratio because he recorded 0 interceptions this year, and since we cant divide by 0, 'NaN' is returned.

#### Sorting results

Now, let's save this result as an object, and further subset the rows to only include player/season combinations where the sum of attempts is at least
50.

Then, 
* Sort the rows descending by completion_percentage and report the first 40 values!
* Sort the rows descending by td_int_ratio and report the first 40 values!

This should give us a good idea of which QB seasons have the best completion percentage and td/int ratio, which are important metrics for determining how accurate and how successful a given QB was for that regular season.


In [36]:
nfl_result_ps = nfl_agg_ps

# Subset to only include player/season combinations with at least 50 attempts
nfl_result_ps = nfl_result_ps[nfl_result_ps["attempts_sum"] >= 50]

# Sort descending by completion_percentage and report first 40
print("Top 40 by Completion Percentage:")
nfl_result_ps.sort_values("completion_percentage", ascending=False).head(40)

Top 40 by Completion Percentage:


,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,0.754717,inf
1248,Colt McCoy,2021,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,0.747475,3.000000
900,Matt Schaub,2019,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,0.746269,3.000000
953,Drew Brees,2018,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1338,Mason Rudolph,2023,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,0.743243,inf
1133,Taysom Hill,2020,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,0.727273,2.000000
1007,Nick Foles,2018,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,0.723077,1.750000
917,Drew Brees,2017,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000


From this output, the top 5 QBs based on completion percentage are:
* 2023 C.J. Beathard, 75.47% completion rate on 53 pass attempts
* 2021 Colt McCoy, 74.74% completion rate on 99 attempts
* 2019 Matt Schaub, 74.62% completion rate on 67 attempts
* 2018 Drew Brees, 74.43% completion rate on 489 attempts
* 2019 Drew Brees, 74.33% completion rate on 378 attempts

This gives us an idea of what the most accurate QB seasons were over this time period, but the discrepancy between pass attempts of these players is alarming. Not to discredit players, but being 74% accurate on n=489 attempts is much more impressive than 75% accurate on n=53 attempts. I believe the restriction of at least 50 attempts still far undershoots what we should filter for to target **starting** NFL quarterbacks, i.e. quarterbacks who played in nearly every game for the majority of the game, over the course of a 16 game season. According to the depth charts, these first three players were not starters, and only filled-in in games when the team had a monumental lead or the starter was injured. We also could have the issue with other observations of starters who played really well for a few games, and then got hurt and missed substantial time. I want to filter for starting QBs who sustained an elite level of accuracy over the course of a regular season.

If we want to filter this for **starting** NFL quarterbacks, we should increase this filter. According to [Statmuse](https://www.statmuse.com/nfl/ask/average-attempts-by-qbs-in-the-last-20-seasons), a reputable source for NFL data, quarterbacks averaged 27 passing attempts per game since 2006, or an **average of 432 pass attempts** for a 16 game season. As an additional exercise, let's adjust the filters to try and get QBs who were starters and played a significant portion of the regular season.

##### Exercise: Sorting for Starting QBs

As a conservative estimate, let's change the filtering parameters to quarterbacks who attempted at least **300** passes in a season, and see who the most accurate seasons were. This should paint a much better picture of the most accurate starting quarterback seasons.

In [49]:
# Subset to only include player/season combinations with at least 300 attempts
nfl_result_ps = nfl_result_ps[nfl_result_ps["attempts_sum"] >= 300]

# Sort descending by completion_percentage and report first 15
print("Top 15 by Completion Percentage (Minimum 300 attempts):")
nfl_result_ps.sort_values("completion_percentage", ascending=False).head(15)

Top 15 by Completion Percentage (Minimum 300 attempts):


,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
953,Drew Brees,2018,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
917,Drew Brees,2017,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000
510,Drew Brees,2011,471,29.437500,660,41.250000,5535.0,345.937500,46,2.875000,14.0,0.875000,0.713636,3.285714
1262,Aaron Rodgers,2020,372,23.250000,526,32.875000,4299.0,268.687500,48,3.000000,5.0,0.312500,0.707224,9.600000
242,Drew Brees,2009,363,24.200000,514,34.266667,4388.0,292.533333,34,2.266667,11.0,0.733333,0.706226,3.090909
1098,Drew Brees,2020,275,22.916667,390,32.500000,2942.0,245.166667,24,2.000000,6.0,0.500000,0.705128,4.000000
1437,Joe Burrow,2021,366,22.875000,520,32.500000,4611.0,288.187500,34,2.125000,14.0,0.875000,0.703846,2.428571
1101,Derek Carr,2019,361,22.562500,513,32.062500,4054.0,253.375000,21,1.312500,8.0,0.500000,0.703704,2.625000


Now, this list is filtered for QBs who played much more! We can see that Drew Brees dominates this list, appearing 7 times! He is often regarded as being one of the most accurate QBs of all time. Sam Bradford, Aaron Rodgers, Joe Burrow, and Derek Carr also had great seasons marked by high accuracy (over 70%) and over 500 attempts each!

In [37]:
# Sort descending by td_int_ratio and report first 40
print("Top 40 by TD/INT Ratio:")
nfl_result_ps.sort_values("td_int_ratio", ascending=False).head(40)

Top 40 by TD/INT Ratio:


,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,inf
26,Matt Schaub,2005,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,0.515625,inf
73,Todd Collins,2007,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,0.638095,inf
455,Troy Smith,2007,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,0.526316,inf
520,Jake Locker,2011,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,0.515152,inf
775,Brian Hoyer,2016,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,0.670000,inf
778,Nick Foles,2016,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,0.654545,inf
812,Derek Anderson,2014,65,13.000000,97,19.400000,701.0,140.200000,5,1.000000,0.0,0.000000,0.670103,inf
940,Jimmy Garoppolo,2016,43,7.166667,63,10.500000,502.0,83.666667,4,0.666667,0.0,0.000000,0.682540,inf
984,Matt Moore,2019,59,9.833333,91,15.166667,659.0,109.833333,4,0.666667,0.0,0.000000,0.648352,inf


At the top of this list, we have 2006 Charlie Batch, who threw for 5 touchdowns, 0 interceptions, and 52 attempts. Again, this list is dominated by players who threw relatively few pass attempts, and we see that the top of the list contains players who threw 0 interceptions over the course of the season, and had a td_int_ratio of *inf* or infinity. This infinity happens because of the quirky way pandas-on-Spark handles dividing by 0, and we'll go into more depth later on.

As an exercise, lets again filter for only QBs who threw at least 300 attempts and recorded at least 1 interception to get an idea of the real starters who recorded legendary seasons with a lot of touchdowns, and few interceptions.

In [50]:
# Subset to only include player/season combinations with at least 300 attempts and at least 1 interception
nfl_result_ps = nfl_result_ps[(nfl_result_ps["attempts_sum"] >= 300)
                              & (nfl_result_ps["interceptions_sum"] >= 1)]
# Sort descending by completion_percentage and report first 15
print("Top 15 by TD/Int Ratio (Minimum 300 attempts and 1 interception):")
nfl_result_ps.sort_values("td_int_ratio", ascending=False).head(15)

Top 15 by TD/Int Ratio (Minimum 300 attempts and 1 interception):


,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
763,Tom Brady,2016,291,24.250000,432,36.000000,3554.0,296.166667,28,2.333333,2.0,0.166667,0.673611,14.000000
639,Nick Foles,2013,203,15.615385,317,24.384615,2891.0,222.384615,27,2.076923,2.0,0.153846,0.640379,13.500000
989,Aaron Rodgers,2018,372,23.250000,597,37.312500,4442.0,277.625000,25,1.562500,2.0,0.125000,0.623116,12.500000
1262,Aaron Rodgers,2020,372,23.250000,526,32.875000,4299.0,268.687500,48,3.000000,5.0,0.312500,0.707224,9.600000
1111,Aaron Rodgers,2021,366,22.875000,531,33.187500,4115.0,257.187500,37,2.312500,4.0,0.250000,0.689266,9.250000
286,Tom Brady,2010,324,20.250000,492,30.750000,3900.0,243.750000,36,2.250000,4.0,0.250000,0.658537,9.000000
695,Aaron Rodgers,2014,341,21.312500,520,32.500000,4381.0,273.812500,38,2.375000,5.0,0.312500,0.655769,7.600000
662,Aaron Rodgers,2011,342,22.800000,501,33.400000,4636.0,309.066667,45,3.000000,6.0,0.400000,0.682635,7.500000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1073,Aaron Rodgers,2019,353,22.062500,569,35.562500,4002.0,250.125000,26,1.625000,4.0,0.250000,0.620387,6.500000


Here we see some truly impressive seasons in terms of throwing many touchdowns, and few interceptions. Tom Brady leads the pack with his 2016 season, throwing 28 touchdowns and only 2 interceptions, a TD/INT ratio of 14! Nick Foles is right under him, throwing 27/2 in 2013. The true king of this stat is Aaron Rodgers, who's name appears 6 times in the top 15 seasons, throwing 25/2 in 2018, 48/5 in 2020, and 37/4 in 2021, showing a ridiculous level of consistency.

### II. Spark SQL

Now, let's repeat the same process for this NFL data using the Spark SQL DataFrame. There are some key syntax differences using Spark SQL instead of pandas-on-Spark, such as:
* Filtering with `.filter` instead of indexing with `[]`
* Aggregating with `F.sum()` instead of `.groupBy().agg`
* Sorting with `.orderBy()` instead of `.sort_values`

#### Load Data

In [38]:
# Read in the weekly NFL data using Spark SQL DataFrame
nfl_spark = spark.read.load("weekly_nfl_data.csv", format="csv", header=True, inferSchema=True)
nfl_spark.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [39]:
print(nfl_spark.columns)

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


#### Subsetting and Variable Creation

In [40]:
# Subset rows: position == "QB", season_type == "REG", season between 2005 and 2023
from pyspark.sql import functions as F

nfl_qb_spark = nfl_spark.filter(
    (F.col("position") == "QB") &
    (F.col("season_type") == "REG") &
    (F.col("season") >= 2005) &
    (F.col("season") <= 2023)
)

# Subset columns
nfl_qb_spark = nfl_qb_spark.select(
    "player_display_name", "season", "week",
    "completions", "attempts", "passing_yards",
    "passing_tds", "interceptions"
)

# For each player/season combination, find the sum and mean of each stat
nfl_agg_spark = nfl_qb_spark.groupBy("player_display_name", "season").agg(
    F.sum("completions").alias("completions_sum"),
    F.mean("completions").alias("completions_mean"),
    F.sum("attempts").alias("attempts_sum"),
    F.mean("attempts").alias("attempts_mean"),
    F.sum("passing_yards").alias("passing_yards_sum"),
    F.mean("passing_yards").alias("passing_yards_mean"),
    F.sum("passing_tds").alias("passing_tds_sum"),
    F.mean("passing_tds").alias("passing_tds_mean"),
    F.sum("interceptions").alias("interceptions_sum"),
    F.mean("interceptions").alias("interceptions_mean")
)

# Create two new variables
nfl_agg_spark = nfl_agg_spark.withColumn(
    "completion_percentage",
    F.col("completions_sum") / F.col("attempts_sum")
)
nfl_agg_spark = nfl_agg_spark.withColumn(
    "td_int_ratio",
    F.col("passing_tds_sum") / F.col("interceptions_sum")
)

nfl_agg_spark.show(5)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum|interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|1.3076923076923077|             11.0|0.8461538461538461|   0.6102088167053364|1.5454545454545454|
|       Jake Plu

#### Sorting results

In [41]:
# Save result, subset to attempts_sum >= 50
nfl_result_spark = nfl_agg_spark.filter(F.col("attempts_sum") >= 50)

# Sort descending by completion_percentage and report first 40
print("Top 40 by Completion Percentage:")
nfl_result_spark.orderBy(F.col("completion_percentage").desc()).show(40)

Top 40 by Completion Percentage:
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|   passing_tds_mean|interceptions_sum| interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|             40| 6.666666666666667|          53| 8.833333333333334|            349.0|58.166666666666664|              1|0.16666666666666666|              0.0|                0.0|   0.754716981132

Note that the format of the output using Spark SQL is different compared to creating a neater pandas dataframe, but we still get the same results for the most accurate QB seasons with at least 50 completions.

In [42]:
# Sort descending by td_int_ratio and report first 40
print("Top 40 by TD/INT Ratio:")
nfl_result_spark.orderBy(F.col("td_int_ratio").desc()).show(40)

Top 40 by TD/INT Ratio:
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum| interceptions_mean|completion_percentage|     td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+
|          Tom Brady|  2016|            291|             24.25|         432|              36.0|           3554.0| 296.1666666666667|             28|2.3333333333333335|              2.0|0.16666666666666666|   0.6736111111111112|           

##### Note: td_int_ratio Division by Zero Behavior

There is an important difference in the results above compared to using Pandas-on-spark. Above, pandas-on-spark included QBs who threw 0 interceptions, and their td_int_ratio was recorded as infinity. Here, using Spark SQL, QBs who threw 0 interceptions are not included!

When a QB has zero interceptions in a season, computing `td_int_ratio` (passing_tds / interceptions) involves division by zero. The two APIs 
handle this differently:

- **pandas-on-Spark** follows standard Python/pandas behavior and returns 
  `inf` (infinity), meaning these player/season combinations sort to the 
  **top** of the descending ranking.
- **Spark SQL** returns `null` for division by zero, meaning these 
  player/season combinations sort to the **bottom** of the descending 
  ranking by default.

This is why the Top 40 TD/INT ratio results differ between the two 
APIs and the pandas-on-Spark method is dominated by players with zero 
interceptions, while Spark SQL shows only players with at least one 
interception ranked by their actual ratio.

### Conclusion

Both of these APIs produced similar results for analyzing NFL quarterbacks, but have some key differences with syntax and handling of data

**pandas-on-Spark** Felt generally a bit easier and more inline with what I'm used to with pandas syntax, although it has its quirks such as returning `inf` for division by 0.

**Spark SQL** Felt more comprehensive and powerful, but more foreign to me. I think its handling of dividing by zero is correct compared to pandas, but I also much prefer the clean tabular output of pandas, as the Spark SQL output is somewhat messy and distorted if there are too many columns for the page width.


